# பாடம் 05 - முகவர் RAG


## அமைப்பு

இந்த நோட்புக் Microsoft முகவர் கட்டமைப்பு பயன்படுத்தி Agentic RAG (Retrieval-Augmented Generation) முறைமையை எளிதாக்குகிறது.

**தேவைப்படும் முன்னேற்பாடுகள்:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — உங்கள் Azure AI Search சேவை Endpoint
- `AZURE_SEARCH_API_KEY` — உங்கள் Azure AI Search API Key
- சூழல் மாறிலிகள் மூலம் கட்டமைக்கப்பட்ட Azure OpenAI விடுவிப்பு
- Azure CLI அங்கீகாரம் பெற்றது (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ஏஜென்டிக் RAG என்றால் என்ன?

பாரம்பரிய RAG ஒரு நிர்ணயிக்கப்பட்ட பணிமுறையை பின்பற்றுகிறது: ஆவணங்களை மீட்டெடுத்து, பிறகு பதில் உருவாக்குகிறது. **ஏஜென்டிக் RAG** மேலும் சென்று, ஏஜெண்டுக்கு எப்போது மற்றும் எப்படி தகவலை மீட்டெடுக்க வேண்டும் என்பதை சுயமாக தீர்மானிக்கும் சுதந்திரத்தை அளிக்கிறது.

ஏஜென்டிக் RAG உடன், ஏஜெண்ட் செய்ய முடியும்:
- ஒரு கேள்விக்கு பதில் அளிக்க முன்னர் தகவல் மீட்டெடுப்பு தேவையானதா என்பதை **தீர்மானிக்க**
- எந்த தரவுத் தளம் அல்லது கருவியை கேள்வி கேட்க **தேர்ந்தெடுக்க**
- மீட்டெடுக்கப்பட்ட முடிவுகளை **மதிப்பீடு செய்து**, முதல் முயற்சி போதுமானதாக இல்லையெனில் தொடர்ச்சியான மீட்டெடுப்புகளை செய்ய
- பல மீட்டெடுக்கல் படிகள் மூலம் பெறப்பட்ட தகவல்களை ஒன்றிணைத்து ஒரு சகதானமான பதிலை **சேர்க்க**

இது ஏஜெண்டை ஒரு நிலையான மீட்டெடுத்து பிறகு உருவாக்கும் பணிமுறையைவிட அதிகமாக இயல்புடையதும் துல்லியமானதுமானது ஆக்குகிறது.


## தேடல் கருவி உருவாக்குதல்

Agentic RAG-இல், வெளிப்புற தரவுக் கூறுகள் முகவர்கள் வேண்டுமானால் அழைக்கக்கூடிய **கருவிகள்**ஆக மையப்படுத்தப்படுகின்றன. இது முகவர் பெறுதலை கட்டாயமான ஒரு படியல்லாமல் எடுத்துக் கொள்ளக்கூடிய இன்னொரு செயல் என்று கருதுவதற்கு இடமாகிறது.

கீழே நாம் பயண அறிவுத்தொகுப்பை வரையறுக்கின்றோம் மற்றும் முகவர் இடம் தொடர்பான தகவல்களைத் தேட அழைக்கக்கூடிய ஒரு கருவியாக அதனை வெளியிடுகின்றோம்.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG முகவரியை உருவாக்குதல்

இப்போது நாம் எப்போதும் பதில் அளிப்பதற்கு முன்பு தகவல்களை **எப்போதும் மீட்டெடுக்கும்** என்று அறிவுறுத்தப்பட்ட ஒரு முகவரியை உருவாக்குகிறோம். முகவர் தன் பதில்களை தன் பயிற்சி தரவுக்கு பதிலாக அறிவுத் தளத்தில் நிலைநிறுத்த `search_travel_knowledge` கருவியை பயன்படுத்துகிறது.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## முறைசார்ந்த மீட்பு — தயாரிப்பவர்-தணிப்பவர் மாதிரியில்

Agentic RAG இந்த முறைசார்ந்த மீட்பில் ஒரு முக்கியமான நன்மை உள்ளது. முகவர் தனது முதல் கண்டுபிடிப்புகளை சரிபார்க்க, துல்லியப்படுத்த அல்லது விரிவுபடுத்த பல முறை தேடல் நடத்த முடியும் — இது "தயாரிப்பவர்-தணிப்பவர்" வேலை முறையைப் போலும்:

1. **தயாரிப்பவர் படி**: முகவர் முதன்மை தகவல்களை மீட்டெடுத்து, பதிலை உருவாக்குகிறது.
2. **தணிப்பவர் படி**: முகவர் விவரங்களைப் பரிசோதிக்க அல்லது இடைவெளிகளை நிரப்ப கூடுதல் தேடல்களைச் செய்கிறது.

கீழே, முகவருக்கு பல இடங்களை ஒப்பிடுதல் தேவையான கேள்வி கேட்கப்படுகிறது, இது அவனை பல முறை தேடச் சுற்றுகிறது.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## சாராம்சம்

இந்த பாடத்தில் நீங்கள் Microsoft Agent Framework-ஐ பயன்படுத்தி **Agentic RAG** அமைப்பை எப்படி உருவாக்குவது என்பதை கற்றுக்கொண்டீர்கள்:

- **Agentic RAG** முகவர்கள் தகவலை எப்போது மீட்டெடுக்க வேண்டும் என்று தானாக முடிவெடுக்க அனுமதிக்கிறது, இதனால் மீட்டெடுத்தல் நிலைத்திருக்காமல் தானாக மாறக்கூடியதாகும்.
- **கருவிகள் தரவു மூலங்களாக**: வெளிப்புற அறிவுத்தளம் (Azure AI Search போல) முகவர் பயன்படுத்தக்கூடிய கருவிகளாக மூடப்பட்டிருப்பது.
- **மறுமுறை மீட்டெடுத்தல்**: தயாரிப்பாளர்-சரிபார்ப்பாளர் முறை முகவருடன் பல மீட்டெடுத்தல் சுற்றுகளை செயல்படுத்த அனுமதிக்கிறது — தேடல், உறுதிப்படுத்தல், மற்றும் படிப்படியான மேம்படுத்தல் — இறுதி பதிலை உருவாக்குவதற்கு முன்பு.

உற்பத்தியில், பெரிய அளவிலான பயண ஆவண மீட்டெடுத்தலை கையாள, நீங்கள் நினைவகத்தில் உள்ள `TRAVEL_KNOWLEDGE_BASE`-ஐ நிஜமான Azure AI Search குறியீட்டுடன் மாற்றுவீர்கள்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
